# Atividade 07.2 — Classificação KNN
## Dataset: Dry Bean Dataset (Multiclasse — 7 classes)

Este notebook adapta o algoritmo KNN para trabalhar com **múltiplas classes** usando o Dry Bean Dataset.  
O dataset possui **13.611 amostras**, **16 features morfológicas** e **7 classes** de feijão:
`SEKER`, `BARBUNYA`, `BOMBAY`, `CALI`, `HOROZ`, `SIRA`, `DERMASON`.

### Mudanças em relação à versão binária:
- Carregamento via `.xlsx` (sem ucimlrepo)
- `LabelEncoder` para converter classes textuais em números
- Busca de K limitada a ímpares de 1 a 31 (dataset grande → evitar lentidão)
- Métricas multiclasse: **matriz de confusão**, **precision**, **recall** e **F1-score por classe**
- Relatório de desempenho por classe


## 0. Instalação e Importações

In [ ]:
!pip install -q scikit-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Bibliotecas importadas com sucesso!")

## 1. Carregamento e Exploração do Dataset

In [ ]:
# Carregar o dataset a partir do arquivo Excel
df = pd.read_excel("Dry_Bean_Dataset.xlsx")

print(f"Shape do dataset: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nClasses encontradas: {sorted(df['Class'].unique())}")
print(f"\nDistribuição das classes:")
print(df['Class'].value_counts())

In [ ]:
# Visualizar distribuição das classes
plt.figure(figsize=(9, 4))
class_counts = df['Class'].value_counts()
bars = plt.bar(class_counts.index, class_counts.values,
               color=plt.cm.Set2.colors[:len(class_counts)])
for bar, val in zip(bars, class_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             str(val), ha='center', va='bottom', fontsize=10)
plt.title('Distribuição das Classes — Dry Bean Dataset', fontsize=13)
plt.xlabel('Classe')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# Verificar valores ausentes
missing = df.isnull().sum()
threshold = 0.5 * len(df)
cols_to_drop = missing[missing > threshold].index.tolist()

print(f"Colunas com >50% ausentes (removidas): {cols_to_drop if cols_to_drop else 'Nenhuma'}")

df_clean = df.drop(columns=cols_to_drop)
print(f"Shape após limpeza: {df_clean.shape}")
print(f"Valores ausentes totais: {df_clean.isnull().sum().sum()}")

## 2. Separação de Features e Target (com LabelEncoder)

> **Multiclasse:** O target (`Class`) é textual. Usamos `LabelEncoder` para converter para inteiros, mantendo o mapeamento original.

In [ ]:
# Separar features e target
X = df_clean.drop(columns=['Class'])
y_raw = df_clean['Class']

# Codificar classes textuais → inteiros
le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Mapeamento de classes (Label → Inteiro):")
for i, cls in enumerate(le.classes_):
    print(f"  {cls:>10} → {i}")

print(f"\nTotal de features: {X.shape[1]}")
print(f"Total de amostras: {len(y)}")
print(f"Número de classes: {len(le.classes_)}")

## 3. Divisão Treino / Validação / Teste (Estratificada)

In [ ]:
# 70% treino, 15% validação, 15% teste (estratificado por classe)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Normalização Min-Max — fit APENAS no treino (sem data leakage)
train_min = X_train.min()
train_range = (X_train.max() - train_min).replace(0, 1)

X_train = (X_train - train_min) / train_range
X_val   = (X_val   - train_min) / train_range
X_test  = (X_test  - train_min) / train_range

n = len(y)
print(f"Treino:    {len(X_train):>5} amostras ({len(X_train)/n:.0%})")
print(f"Validação: {len(X_val):>5} amostras ({len(X_val)/n:.0%})")
print(f"Teste:     {len(X_test):>5} amostras ({len(X_test)/n:.0%})")

## 4. Implementação do KNN (compatível com Multiclasse)

> O algoritmo usa **votação por maioria** entre os K vizinhos mais próximos.  
> Como `np.unique` retorna todas as classes presentes, a predição funciona naturalmente para qualquer número de classes.

In [ ]:
class KNN:
    """K-Nearest Neighbors — funciona para binário E multiclasse por votação."""

    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._predict(x) for x in X])

    def _predict(self, x):
        # Distância Euclidiana para todos os pontos de treino
        distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
        # Índices dos K vizinhos mais próximos
        k_indices = np.argsort(distances)[:self.k]
        k_labels  = self.y_train[k_indices]
        # Votação: retorna a classe com mais votos
        classes, counts = np.unique(k_labels, return_counts=True)
        return classes[np.argmax(counts)]


def accuracy_score_manual(y_true, y_pred):
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    return np.mean(y_true == y_pred)


print("Classe KNN e função de acurácia definidas com sucesso!")

## 5. Métricas Multiclasse

Para multiclasse, além da acurácia, calculamos **Precision**, **Recall** e **F1-score** por classe (macro average).

In [ ]:
def confusion_matrix_manual(y_true, y_pred, n_classes):
    """Matriz de confusão NxN para multiclasse."""
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm


def classification_report_manual(y_true, y_pred, class_names):
    """Precision, Recall e F1 por classe + macro average."""
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()
    n_classes = len(class_names)
    cm = confusion_matrix_manual(y_true, y_pred, n_classes)

    precisions, recalls, f1s = [], [], []
    header = f"{'Classe':>12}  {'Precision':>10}  {'Recall':>8}  {'F1-Score':>9}  {'Support':>8}"
    print(header)
    print("-" * len(header))

    for i, cls in enumerate(class_names):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

        print(f"{cls:>12}  {precision:>10.4f}  {recall:>8.4f}  {f1:>9.4f}  {support:>8}")

    print("-" * len(header))
    print(f"{'macro avg':>12}  {np.mean(precisions):>10.4f}  {np.mean(recalls):>8.4f}  {np.mean(f1s):>9.4f}  {len(y_true):>8}")
    print(f"\nAcurácia geral: {accuracy_score_manual(y_true, y_pred):.4f}")


def plot_confusion_matrix(y_true, y_pred, class_names, title="Matriz de Confusão"):
    cm = confusion_matrix_manual(y_true, y_pred, len(class_names))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title, fontsize=13)
    plt.ylabel('Real')
    plt.xlabel('Previsto')
    plt.tight_layout()
    plt.show()


print("Funções de métricas multiclasse definidas!")

## 6. Busca pelo Melhor K (no conjunto de Validação)

> O dataset possui 13.611 amostras. Por eficiência, testamos apenas **K ímpares de 1 a 31**.  
> A regra de usar apenas ímpares evita empates na votação.

In [ ]:
# Testamos K ímpares de 1 a 31 (eficiência para dataset grande)
k_values = [k for k in range(1, 32) if k % 2 != 0]
val_scores = []

print(f"Testando K = {k_values}\n")
print(f"{'K':>4}  {'Acurácia Validação':>20}")
print("-" * 30)

for k in k_values:
    model = KNN(k=k)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc   = accuracy_score_manual(y_val, preds)
    val_scores.append(acc)
    print(f"{k:>4}  {acc:>20.4f}")

best_k  = k_values[np.argmax(val_scores)]
best_acc = max(val_scores)
print(f"\n✔ Melhor K: {best_k}  |  Acurácia na validação: {best_acc:.4f}")

In [ ]:
# Gráfico: Acurácia x K
plt.figure(figsize=(10, 4))
plt.plot(k_values, val_scores, 'o-', color='steelblue', linewidth=2, markersize=6)
plt.axvline(best_k, color='crimson', linestyle='--', label=f'Melhor K = {best_k}')
plt.scatter([best_k], [best_acc], color='crimson', zorder=5, s=80)
plt.title('Acurácia na Validação por Valor de K — KNN Multiclasse', fontsize=12)
plt.xlabel('K (número de vizinhos)')
plt.ylabel('Acurácia')
plt.xticks(k_values)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Avaliação Final no Conjunto de Teste

In [ ]:
# Treino final: TREINO + VALIDAÇÃO após escolha do melhor K
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = np.concatenate([y_train, y_val])

final_model = KNN(k=best_k)
final_model.fit(X_train_final, y_train_final)

# Predição no TESTE (nunca visto antes)
y_pred = final_model.predict(X_test)

final_acc = accuracy_score_manual(y_test, y_pred)
print(f"\n{'='*45}")
print(f"   RESULTADO FINAL — Conjunto de Teste")
print(f"{'='*45}")
print(f"   Melhor K:        {best_k}")
print(f"   Acurácia Final:  {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"{'='*45}")

In [ ]:
# Relatório de métricas por classe
print("\n--- Relatório de Classificação (Multiclasse) ---\n")
classification_report_manual(y_test, y_pred, le.classes_)

In [ ]:
# Matriz de confusão
plot_confusion_matrix(
    y_test, y_pred, le.classes_,
    title=f"Matriz de Confusão — KNN (k={best_k}) — Dry Bean Dataset"
)

## 8. Verificação de Robustez (Múltiplos random_state)

In [ ]:
def run_knn_pipeline(X_data, y_data, rs=42, k_max=31):
    """Pipeline completo: split → normalização → busca K → treino final → avaliação."""

    # Split estratificado
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_data, y_data, test_size=0.30, random_state=rs, stratify=y_data
    )
    X_vl, X_ts, y_vl, y_ts = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=rs, stratify=y_tmp
    )

    # Normalização Min-Max (fit no treino)
    t_min = X_tr.min()
    t_rng = (X_tr.max() - t_min).replace(0, 1)
    X_tr = (X_tr - t_min) / t_rng
    X_vl = (X_vl - t_min) / t_rng
    X_ts = (X_ts - t_min) / t_rng

    # Busca do melhor K na validação
    k_vals = [k for k in range(1, k_max + 1) if k % 2 != 0]
    val_accs = []
    for k in k_vals:
        m = KNN(k=k)
        m.fit(X_tr, y_tr)
        val_accs.append(accuracy_score_manual(y_vl, m.predict(X_vl)))

    bk = k_vals[np.argmax(val_accs)]

    # Treino final (treino + validação)
    X_tr_f = pd.concat([X_tr, X_vl], axis=0)
    y_tr_f = np.concatenate([y_tr, y_vl])
    fm = KNN(k=bk)
    fm.fit(X_tr_f, y_tr_f)

    test_acc = accuracy_score_manual(y_ts, fm.predict(X_ts))
    return {"best_k": bk, "val_acc": max(val_accs), "test_acc": test_acc}


random_states = [0, 7, 21, 42, 99]
results = []

print(f"{'random_state':>14}  {'best_k':>8}  {'val_acc':>10}  {'test_acc':>10}")
print("-" * 50)

for rs in random_states:
    r = run_knn_pipeline(X, y, rs=rs)
    results.append(r)
    print(f"{rs:>14}  {r['best_k']:>8}  {r['val_acc']:>10.4f}  {r['test_acc']:>10.4f}")

test_accs = [r['test_acc'] for r in results]
print(f"\n{'='*50}")
print(f"Acurácia média: {np.mean(test_accs):.4f}")
print(f"Desvio padrão:  {np.std(test_accs):.4f}")
print(f"Ks selecionados: {[r['best_k'] for r in results]}")

## 9. Resumo

| Aspecto | Detalhe |
|---|---|
| Dataset | Dry Bean Dataset (13.611 amostras, 16 features) |
| Classes | 7 (SEKER, BARBUNYA, BOMBAY, CALI, HOROZ, SIRA, DERMASON) |
| Algoritmo | KNN com votação por maioria |
| Normalização | Min-Max (fit apenas no treino) |
| Seleção de K | K ímpares de 1 a 31 por validação |
| Métricas | Acurácia, Precision, Recall, F1-Score por classe, Matriz de Confusão |

### Por que o KNN funciona para multiclasse sem modificação?

O passo de **votação por maioria** (`np.unique` + `np.argmax`) generaliza naturalmente para qualquer número de classes.  
A distância Euclidiana e a seleção dos K vizinhos são independentes do número de classes — apenas o rótulo com mais votos vence.